# Datathon 2026 — Part 2: Exploratory Data Analysis
**VinTelligence × VinUniversity DS&AI Club**

Analysis follows the **Descriptive → Diagnostic → Predictive → Prescriptive** framework as required by the rubric.


## 1. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

## 2. Data Loading

In [ ]:
DATA_PATH = "dataset/"

files = {}
for f in os.listdir(DATA_PATH):
    if f.endswith(".csv"):
        name = f.replace(".csv", "")
        files[name] = pd.read_csv(DATA_PATH + f)
        print(f"{f:35s} → {files[name].shape}")


## 3. Preprocessing

In [ ]:
# ── Parse dates ───────────────────────────────────────────────
sales       = files['sales'].copy()
orders      = files['orders'].copy()
order_items = files['order_items'].copy()
products    = files['products'].copy()
customers   = files['customers'].copy()
geography   = files['geography'].copy()
returns     = files['returns'].copy()
promotions  = files['promotions'].copy()
web_traffic = files['web_traffic'].copy()
inventory   = files['inventory'].copy()

sales['Date']              = pd.to_datetime(sales['Date'])
orders['order_date']       = pd.to_datetime(orders['order_date'])
returns['return_date']     = pd.to_datetime(returns['return_date'])
promotions['start_date']   = pd.to_datetime(promotions['start_date'])
promotions['end_date']     = pd.to_datetime(promotions['end_date'])
web_traffic['date']        = pd.to_datetime(web_traffic['date'])
inventory['snapshot_date'] = pd.to_datetime(inventory['snapshot_date'])

# ── Derived columns ───────────────────────────────────────────
sales['gross_margin'] = (sales['Revenue'] - sales['COGS']) / sales['Revenue']
sales['year']         = sales['Date'].dt.year
sales['month']        = sales['Date'].dt.month
sales['day_of_week']  = sales['Date'].dt.day_name()

# ── Promotion active flag (vectorized) ────────────────────────
date_range = pd.date_range(sales['Date'].min(), sales['Date'].max(), freq='D')
promo_active_map = pd.Series(False, index=date_range)
for _, row in promotions.iterrows():
    mask = (date_range >= row['start_date']) & (date_range <= row['end_date'])
    promo_active_map[mask] = True
sales['promo_active'] = sales['Date'].map(promo_active_map)

# ── Merged order table (safe version — no duplicate rows) ─────
return_summary = returns.groupby('order_id').agg(
    return_reason=('return_reason', 'first'),
    total_returned=('return_quantity', 'sum')
).reset_index()

review_summary = files['reviews'].groupby('order_id').agg(
    rating=('rating', 'mean')
).reset_index()

merged_order = (
    orders
    .merge(order_items,                                on='order_id',    how='left')
    .merge(products[['product_id','category','segment','price','cogs']], on='product_id', how='left')
    .merge(customers[['customer_id','gender','age_group','acquisition_channel']], on='customer_id', how='left')
    .merge(geography[['zip','city','region']],         on='zip',         how='left')
    .merge(review_summary,                             on='order_id',    how='left')
    .merge(return_summary,                             on='order_id',    how='left')
)

merged_order['revenue']            = merged_order['unit_price'] * merged_order['quantity']
merged_order['return_reason']      = merged_order['return_reason'].fillna('Not Returned')
merged_order['rating']             = merged_order['rating'].fillna(0)
merged_order['discount_pct']       = merged_order['discount_amount'] / (merged_order['unit_price'] + merged_order['discount_amount'])
merged_order['order_date']         = pd.to_datetime(merged_order['order_date'])
merged_order['month']              = merged_order['order_date'].dt.month_name()
merged_order['day_of_week']        = merged_order['order_date'].dt.day_name()

print(f"sales:         {sales.shape}")
print(f"merged_order:  {merged_order.shape}")
print("✅ Preprocessing complete")


## 4. Exploratory Data Analysis

All charts are annotated with their analytical level: **[D]** Descriptive · **[Dx]** Diagnostic · **[P]** Predictive · **[Px]** Prescriptive


### Chart 1 — Monthly Revenue Heatmap `[D → Px]`

In [ ]:
monthly = sales.groupby(['year', 'month'])['Revenue'].sum().reset_index()
pivot   = monthly.pivot(index='year', columns='month', values='Revenue')
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig = px.imshow(
    pivot,
    color_continuous_scale='Greens',
    aspect='auto',
    title='<b>Monthly Revenue Heatmap (2012–2022)</b>'
          '<br><sup>Darker green = higher revenue | reveals seasonality & year-over-year trend</sup>',
    labels=dict(x='Month', y='Year', color='Revenue (VND)')
)
fig.update_layout(width=950, height=450, coloraxis_colorbar_title='Revenue')
fig.show()


**Descriptive — What happened?**
Revenue was weak in the early years (2012–2013, light green throughout) and grew steadily to peak intensity in 2014–2018, where April–June cells are the darkest green in the entire chart — reaching 200–270M VND per month. From 2019 onward the entire heatmap fades to near-white, with most months falling below 100M VND by 2022.

**Diagnostic — Why did it happen?**
The April–June column is consistently the darkest across almost every year — confirming Q2 as a structural seasonal peak driven by the spring/summer fashion cycle. The August column shows a secondary darker band in 2014–2018, suggesting a mid-year restocking wave. The 2019 fade predates COVID-19, pointing to a structural issue — likely competitive displacement from Shopee/Lazada — rather than a purely external shock.

**Predictive — What is likely to happen?**
No re-darkening trend is visible through 2022 — the heatmap stays uniformly pale in the bottom rows. The 2023–2024 forecast should be anchored to the post-2019 low-revenue regime. The seasonal shape (darkest in Apr–Jun) is likely to persist even at lower absolute levels.

**Prescriptive — What should we do?**
The stark contrast between the 2014–2018 dark rows and the 2020–2022 near-white rows quantifies the scale of the revenue collapse. Conduct a root-cause analysis of the 2019 inflection by comparing 2018 vs 2019 cohort retention rates. A win-back campaign targeting customers whose last order was pre-2019 could recover meaningful revenue at low acquisition cost, and should be timed to launch before April to capitalise on the seasonal peak.

### Chart 2 — Average Monthly Orders (Year-over-Year) `[D → Px]`

In [ ]:
monthly_orders = (
    orders.assign(month=orders['order_date'].dt.month_name(),
                  year=orders['order_date'].dt.year)
    .groupby(['year','month'])
    .size()
    .reset_index(name='order_count')
)
avg_monthly = monthly_orders.groupby('month')['order_count'].mean().reset_index()

MONTH_ORDER = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
avg_monthly['month'] = pd.Categorical(avg_monthly['month'], categories=MONTH_ORDER, ordered=True)
avg_monthly = avg_monthly.sort_values('month')

fig = px.bar(
    avg_monthly, x='month', y='order_count', color='month',
    title='<b>Average Monthly Orders (Year-over-Year)</b>'
          '<br><sup>Controls for dataset length bias; shows true seasonal demand pattern</sup>',
    labels=dict(order_count='Avg Orders', month='Month')
)
fig.update_layout(showlegend=False, width=950, height=480)
fig.show()


**Descriptive:** April leads at ~7,433 average orders per year, with May close behind at ~7,000. January is the clear trough at ~3,000 — less than half the April peak. The pattern is bimodal — a major Q2 peak (April–June) and a secondary bump in August (~6,000), before a steady decline through November (~3,200) and a small December recovery (~4,700).

**Diagnostic:** The April–June peak aligns with the Vietnamese spring/summer fashion cycle and end-of-school-year spending. October–November significantly underperforms global e-commerce norms (11.11, Black Friday) — these campaigns are not resonating with this customer base.

**Predictive:** The Q2 peak is the most reliable demand signal in the dataset. Inventory and logistics must be scaled up by late February to capture it.

**Prescriptive:** Aggressively invest in 11.11 and Black Friday to capture the Oct–Nov gap. Avoid deep promotions in April–June — demand is already at its natural peak and promotions there only sacrifice margin.

### Chart 3 — Order & Revenue Distribution by Category × Segment `[D → Px]`

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Order Distribution by Category & Segment',
    'Revenue Distribution by Category & Segment'
])

hm_orders = px.density_heatmap(
    merged_order, x='category', y='segment',
    color_continuous_scale='Greens'
).update_traces(histnorm='percent', texttemplate='%{z:.2f}%')

hm_revenue = px.density_heatmap(
    merged_order, x='category', y='segment',
    z='revenue', histfunc='sum',
    color_continuous_scale='Greens'
).update_traces(histnorm='percent', texttemplate='%{z:.2f}%')

for trace in hm_orders.data:
    fig.add_trace(trace, row=1, col=1)
for trace in hm_revenue.data:
    fig.add_trace(trace, row=1, col=2)

fig.update_layout(
    height=520, width=1050,
    coloraxis=dict(colorscale='Greens'),
    title='<b>Order & Revenue Distribution by Category × Segment</b>'
          '<br><sup>Compare both heatmaps: same cell size ≠ same revenue contribution</sup>'
)
fig.show()


**Descriptive:** Outdoor × Activewear dominates orders at 32.04%, but Streetwear × Everyday (32.72%) and Streetwear × Balanced (31.21%) together account for ~64% of all revenue.

**Diagnostic:** Outdoor × Activewear has 32% of orders but only 12.26% of revenue — very low average order value. Streetwear segments have ~40% combined order share (25.54% Everyday + 14.46% Balanced) but generate 64% of revenue, meaning customers spend significantly more per Streetwear order.

**Predictive:** Revenue is dangerously concentrated in Streetwear. Any trend shift or supply disruption in that category would devastate ~64% of total revenue with no other category large enough to compensate.

**Prescriptive:** Grow Outdoor × Activewear's average order value through bundling and upsells. Guarantee Streetwear inventory depth as the #1 priority. Evaluate discontinuing the near-zero Premium, Standard and All-weather segments.

### Chart 4 — Return Counts by Category × Segment `[D → Px]`

In [ ]:
fig = px.density_heatmap(
    merged_order[merged_order['return_reason'] != 'Not Returned'],
    x='category', y='segment',
    text_auto=True,
    color_continuous_scale='Reds',
    title='<b>Total Return Counts by Category × Segment</b>'
          '<br><sup>Cross-reference with Chart 3 to identify disproportionate return rates</sup>'
)
fig.update_layout(width=750, height=500)
fig.show()


**Descriptive:** Outdoor × Activewear has 13,005 returns — the single darkest cell — followed by Streetwear × Everyday (10,143). Streetwear × Balanced (5,613) and GenZ × Performance (5,442) are the next largest. Casual and GenZ Trendy segments have minimal returns.

**Diagnostic:** Outdoor × Activewear's returns are disproportionate to its revenue contribution (32% of orders, 12.26% revenue, but the highest return volume). Sizing complexity in activewear is the likely driver, confirmed by Chart 8.

**Predictive:** Without intervention, return volumes will grow proportionally with sales, turning an already low-margin category into a net-loss segment once reverse logistics costs are accounted for.

**Prescriptive:** Implement size recommendation tools specifically for Activewear. Introduce an exchange-first return policy to reduce cash refund outflow. Use Casual and GenZ's low return rates as an internal benchmark.

### Chart 5 — Age Group Revenue Distribution `[D → Px]`

In [ ]:
fig = (
    px.histogram(
        merged_order.dropna(subset=['age_group']).sort_values('age_group'),
        x='age_group', y='revenue',
        histnorm='percent',
        range_y=[0, 40],
        title='<b>Revenue Distribution by Age Group</b>'
              '<br><sup>Shows which demographic drives the most revenue</sup>',
        labels=dict(revenue='% of Revenue', age_group='Age Group'),
        color_discrete_sequence=['#636EFA','#EF553B','#00CC96','#AB63FA','#FFA15A']
    )
    .update_traces(texttemplate='%{y:.1f}%', textposition='outside', marker_color=['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A'])
    .update_layout(showlegend=False, width=750, height=480)
)
fig.show()


**Descriptive:** 25–34 leads at 29.5%, followed by 35–44 (26.3%), 45–54 (19.3%), 18–24 (13.7%), and 55+ (11.2%). The 25–44 bracket combined accounts for 55.8% of all revenue.

**Diagnostic:** The 18–24 group underperforms its population size — digitally native but lower disposable income. The 55+ group's 11.2% is notable and suggests older demographics are more engaged than expected.

**Predictive:** Vietnam's current 25–34 cohort will age into 35–44, likely maintaining spending. Acquiring 18–24 now at lower CAC is a long-term investment that pays off in 5–8 years.

**Prescriptive:** Focus retention programs (loyalty, personalised recommendations) on 25–44. Offer installment payment options for 18–24 to lower conversion barriers. Develop a curated product line for 55+ — likely high AOV and low return rates.


### Chart 6 — Promotion Days vs. Non-Promotion Days `[D → Px]`

In [ ]:
promo_compare = (
    sales.groupby('promo_active')
    .agg(avg_revenue=('Revenue','mean'), avg_margin=('gross_margin','mean'))
    .reset_index()
)
promo_compare['label'] = promo_compare['promo_active'].map({True:'Promo Day', False:'Non-Promo Day'})

COLORS = ['#43A047', '#E53935']  # green=promo, red=non-promo
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Average Daily Revenue', 'Average Gross Margin %'))

fig.add_trace(go.Bar(x=promo_compare['label'], y=promo_compare['avg_revenue'],
    marker_color=COLORS, showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=promo_compare['label'], y=promo_compare['avg_margin'],
    marker_color=COLORS, showlegend=False), row=1, col=2)

fig.update_yaxes(tickformat='.1%', row=1, col=2)
fig.update_layout(
    title='<b>Promotion Days vs. Non-Promotion Days</b>'
          '<br><sup>The single most important finding: promotions are revenue-neutral but destroy 83% of gross margin</sup>',
    width=850, height=480
)
fig.show()


**Descriptive:** Non-promo days average ~4.52M VND daily revenue vs ~4M on promo days — promo days actually generate *less* revenue on average. Gross margin on promo days collapses to ~3.5% vs ~18% on non-promo days.

**Diagnostic:** Promotions are **revenue-neutral but margin-destructive**. Customers who would have bought anyway are simply buying at a lower price. At 3.5% gross margin, promo days are almost certainly operating at a net loss after operating costs.

**Predictive:** If promotions run ~20–25% of days annually, they are suppressing overall business margin by several percentage points every year — a compounding drag on profitability.

**Prescriptive:** (1) Set a hard cap of 30% maximum discount depth immediately. (2) Run a hold-out experiment — eliminate promotions for one quarter in a segment of customers and test whether revenue actually drops. (3) Redirect promotion budget toward loyalty rewards for high-value customers rather than blanket sitewide discounts.

### Chart 7 — Gross Margin & Promotions Over Time `[D → Px]`

In [ ]:
sales_sorted = sales.sort_values('Date').copy()
sales_sorted['gross_margin_pct'] = sales_sorted['gross_margin'] * 100

promotions['duration_ms'] = (
    (promotions['end_date'] - promotions['start_date']).dt.total_seconds() * 1000
)

promotions['label'] = promotions.apply(
    lambda r: f"{int(r['discount_value'])}%"
    if 'percent' in str(r['promo_type']).lower() else '',
    axis=1
)

x_min = sales_sorted['Date'].min() - pd.DateOffset(months=2)
x_max = sales_sorted['Date'].max() + pd.DateOffset(months=2)

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Scatter(
    x=sales_sorted['Date'],
    y=sales_sorted['gross_margin_pct'],
    name='Gross Margin',
    mode='lines',
    line=dict(color='#4361EE', width=2),
    fill='tozeroy',
    fillcolor='rgba(67, 97, 238, 0.07)'
), secondary_y=False)

# Break-even line — no built-in annotation, use add_shape + separate add_annotation
fig.add_shape(
    type='line',
    xref='paper', yref='y',
    x0=0, x1=1, y0=0, y1=0,
    line=dict(color='rgba(200,0,0,0.4)', width=1.2, dash='dash')
)
fig.add_annotation(
    xref='paper', yref='y',
    x=0.01,        # slightly inside the plot area, away from the y-axis
    y=1.5,         # just above the zero line
    text='Break-even',
    showarrow=False,
    font=dict(size=10, color='rgba(200,0,0,0.6)', family='Arial'),
    xanchor='left'
)

fig.add_trace(go.Bar(
    x=promotions['start_date'],
    y=promotions['discount_value'],
    width=promotions['duration_ms'],
    offset=0,
    opacity=0.45,
    marker_color='#2DC653',
    marker_line_width=0,
    name='Promotions',
    text=promotions['label'],
    textposition='outside',
    textfont=dict(size=9, color='#1a7a35', family='Arial'),
), secondary_y=True)

fig.update_layout(
    title=dict(
        text='<b>Gross Margin and Promotions Over Time</b>'
             '<br><sup>Daily gross margin (%) vs active promotion discount depth — '
             'margin crashes align with deepest discount campaigns</sup>',
        font=dict(size=16, family='Arial', color='#1a1a2e'),
    ),
    xaxis=dict(
        tickvals=pd.date_range(
            start=sales_sorted['Date'].min().to_period('Q').to_timestamp(),
            end='2022-12-01',
            freq='QS'
        ),
        tickformat='%b %Y',
        tickangle=-45,
        tickfont=dict(size=11, family='Arial', color='#444'),
        showgrid=True,
        gridcolor='rgba(200,200,200,0.3)',
        range=[x_min, x_max]
    ),
    yaxis=dict(
        ticksuffix='%',
        tickfont=dict(size=11, family='Arial', color='#4361EE'),
        title=dict(text='Gross Margin (%)', font=dict(size=12, color='#4361EE')),
        showgrid=True,
        gridcolor='rgba(200,200,200,0.3)',
        zeroline=False,
    ),
    yaxis2=dict(
        ticksuffix='%',
        tickfont=dict(size=11, family='Arial', color='#1a7a35'),
        title=dict(text='Discount Depth (%)', font=dict(size=12, color='#1a7a35')),
        showgrid=False,
    ),
    legend=dict(
        orientation='v',
        x=1.06, y=1,
        font=dict(size=12, family='Arial'),
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='rgba(200,200,200,0.5)',
        borderwidth=1
    ),
    plot_bgcolor='rgba(248,249,252,1)',
    paper_bgcolor='white',
    hovermode='x unified',
    width=1200, height=540,
    bargap=0,
    margin=dict(t=90, l=70, r=100, b=80)
)

fig.show()

**Descriptive — What happened?**
Daily gross margin hovers between 15–25% during normal periods, but crashes sharply and periodically down to the -40% to -60% range. These major crashes are short-lived and occur exactly every 24 months. The green promotion bars show two distinct tiers of discount depth — the vast majority of campaigns run at 10–20%, while a handful reach exactly 50%.

**Diagnostic — Why did it happen?**
Every major margin crash aligns precisely with the tallest green bars — the 50% discount campaigns. While regular 10–20% promotions frequently pull the gross margin temporarily below the break-even line, the catastrophic collapses only occur when the discount hits the 50% mark, causing the business to sell products deeply below their cost of goods.

**Predictive — What is likely to happen?**
The pattern is highly regular — a deep, margin-destroying 50% promotion appears every 24 months, specifically occurring in August. While the late-2022 trend shows margin volatility dipping below 0%, the historical cadence dictates that the next extreme margin crash will not occur until August 2023 if this biennial promotion schedule continues unchanged.

**Prescriptive — What should we do?**
Set a hard cap of 20% maximum discount depth across all campaigns. The data shows that standard 10–20% promotions only cause minor negative-margin dips, while the severe structural damage comes exclusively from the 50% outliers. Eliminating these specific August anomalies would prevent the extreme margin crashes while preserving the vast majority of the normal promotional calendar.

### Chart 8 — Return Reasons by Product Category `[D → Px]`

In [ ]:
returns_full = returns.merge(products[['product_id','category']], on='product_id', how='left')
return_reasons = returns_full.groupby(['category','return_reason'])['return_quantity'].sum().reset_index()

fig = px.bar(
    return_reasons,
    x='category', y='return_quantity', color='return_reason',
    barmode='stack',
    title='<b>Return Reasons by Product Category</b>'
          '<br><sup>wrong_size → size guides | defective → supplier QC | not_as_described → product copy</sup>',
    labels=dict(return_quantity='Returned Units', category='Category')
)
fig.update_layout(width=1000, height=520)
fig.show()


**Descriptive:** Streetwear leads with ~60k returned units, Outdoor ~40k. `wrong_size` (orange) is the #1 reason in both, making up ~35–40% of each bar. `defective` is second, then `not_as_described`, `late_delivery`, and `changed_mind`.

**Diagnostic:** `wrong_size` dominance is a clear operational failure — size guides are inadequate for Streetwear and Outdoor's complex fits. `defective` as #2 in Streetwear (~12k units) points to a supplier quality control problem, not customer behaviour.

**Predictive:** Return volumes will grow proportionally with revenue recovery. Each percentage point reduction in Streetwear's return rate saves thousands of units in reverse logistics annually.

**Prescriptive:** (1) Size quiz tool for Streetwear/Outdoor. (2) Defect rate penalty clauses in top Streetwear supplier contracts. (3) Audit Outdoor product photography for material and fit accuracy. Use Casual/GenZ's near-zero returns as the internal quality benchmark.


### Chart 9 — Stockout Rate vs. Monthly Revenue `[D → Px]`

In [ ]:
inventory['year']      = inventory['snapshot_date'].dt.year
inventory['month_num'] = inventory['snapshot_date'].dt.month

stockout_monthly = (
    inventory.groupby(['year','month_num'])
    .agg(total_products=('product_id','count'), stockout_products=('stockout_flag','sum'))
    .reset_index()
)
stockout_monthly['stockout_rate'] = stockout_monthly['stockout_products'] / stockout_monthly['total_products']
stockout_monthly['date'] = pd.to_datetime(
    stockout_monthly[['year','month_num']].rename(columns={'month_num':'month'}).assign(day=1)
)

monthly_rev = (
    sales.groupby(sales['Date'].dt.to_period('M'))['Revenue'].sum()
    .reset_index()
)
monthly_rev['date'] = monthly_rev['Date'].dt.to_timestamp()
merged_stock = stockout_monthly.merge(monthly_rev[['date','Revenue']], on='date', how='inner')

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(
    x=merged_stock['date'], y=merged_stock['stockout_rate'],
    name='Stockout Rate', mode='lines+markers', line=dict(color='crimson', width=2)
), secondary_y=False)
fig.add_trace(go.Bar(
    x=merged_stock['date'], y=merged_stock['Revenue'],
    name='Monthly Revenue', marker_color='steelblue', opacity=0.5
), secondary_y=True)

fig.update_layout(
    title='<b>Stockout Rate vs. Monthly Revenue</b>'
          '<br><sup>Key finding: stockouts and revenue are uncorrelated — inventory did NOT cause the revenue collapse</sup>',
    width=1100, height=500, hovermode='x unified',
    xaxis=dict(
        range=[sales['Date'].min() - pd.DateOffset(months=3), sales['Date'].max() + pd.DateOffset(months=1)],
        rangemode='normal'
    )
)
fig.update_yaxes(title_text='Stockout Rate', tickformat='.0%', secondary_y=False)
fig.update_yaxes(title_text='Revenue (VND)', secondary_y=True)
fig.show()

**Descriptive:** Stockout rate oscillates between 58–72% throughout the entire 2012–2022 period. Revenue peaked 2016–2018 at 250–270M VND/month and collapsed post-2019 to 50–150M VND — while stockout rate remained unchanged.

**Diagnostic:** The two lines move **independently**. The revenue collapse was not caused by stockouts — inventory problems existed even during the high-revenue years and customers still bought. The real driver must lie elsewhere (competition, product fit, demand decline).

**Predictive:** Fixing stockouts alone will not recover revenue. Both problems — chronic inventory inefficiency and weakening demand — need to be addressed separately.

**Prescriptive:** Set a target to reduce stockout rate below 40% within 12 months to protect brand trust, but treat this as separate from the revenue recovery problem. Investigate the 2019 revenue inflection through customer cohort analysis.


### Chart 10 — Revenue by Geography `[D → Px]`

In [ ]:
orders_geo = orders.merge(geography[['zip', 'city']], on='zip', how='left')
orders_items_geo = order_items.merge(orders_geo[['order_id', 'city']], on='order_id', how='left')
orders_items_geo['line_revenue'] = orders_items_geo['quantity'] * orders_items_geo['unit_price']

city_rev = (orders_items_geo.groupby(['city'])['line_revenue'].sum()
            .reset_index().sort_values('line_revenue', ascending=False).head(15))

fig = go.Figure()

fig.add_trace(go.Bar(
    x=city_rev['line_revenue'], y=city_rev['city'],
    orientation='h', marker_color='steelblue', showlegend=False
))

fig.update_layout(
    title='<b>Top 15 Cities by Revenue</b><br><sup>Provincial cities rival major cities, clustering tightly between 400-581M VND</sup>',
    width=1100, height=520
)

fig.show()

**Descriptive — What happened?** 
The top 15 cities cluster tightly between 400–581M VND — Son Tay leads at 581M, with Hanoi appearing mid-list alongside provincial cities like Kon Tum and Lao Cai.

**Diagnostic — Why did it happen?** 
Small provincial cities matching Hanoi's revenue likely reflects a small number of high-value bulk buyers in those locations. The tight city clustering means no dangerous over-reliance on a single city.

**Predictive — What is likely to happen?** 
As e-commerce penetration grows in industrial zones (Bac Ninh, Bac Giang, Viet Tri), their contribution is likely to increase.

**Prescriptive — What should we do?** 
Investigate high-revenue provincial cities for bulk buyer relationships — assign dedicated account managers if confirmed.

## 5. Variance Analysis

Before reading these charts, here are the three concepts used:

**Variance** measures how spread out your daily values are around their average. A high variance means daily revenue swings wildly between very high and very low values — the business is unpredictable. A low variance means daily revenue is consistently close to the average. Mathematically it is the average of squared deviations from the mean.

**Rolling Variance** applies that calculation inside a sliding time window (here 30 days). Instead of one number for the whole dataset, you get a line that shows *when* the business was volatile vs stable over time. A spike on the rolling variance chart means "during this 30-day period, daily values were all over the place." A flat low period means "revenue was consistent and predictable."

**Coefficient of Variation (CV)** = standard deviation ÷ mean. It is a scale-independent measure of volatility — it answers "how large are the swings *relative to the average level*?" This lets you fairly compare Revenue and COGS even though they are on different scales. A CV of 0.4 means the typical daily swing is about 40% of the average value. CV is more useful than raw variance when comparing two series of different magnitudes.

### Chart 11 — Rolling Variance Over Time `[D → Px]`

In [ ]:
WINDOW = 30
sales_sorted = sales.sort_values('Date').copy()
sales_sorted['rev_var']  = sales_sorted['Revenue'].rolling(WINDOW).var()
sales_sorted['cogs_var'] = sales_sorted['COGS'].rolling(WINDOW).var()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('Revenue — 30-Day Rolling Variance', 'COGS — 30-Day Rolling Variance'),
    vertical_spacing=0.1)

fig.add_trace(go.Scatter(
    x=sales_sorted['Date'], y=sales_sorted['rev_var'],
    mode='lines', name='Revenue Variance',
    line=dict(color='#2196F3', width=1.5),
    fill='tozeroy', fillcolor='rgba(33,150,243,0.1)'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=sales_sorted['Date'], y=sales_sorted['cogs_var'],
    mode='lines', name='COGS Variance',
    line=dict(color='#F44336', width=1.5),
    fill='tozeroy', fillcolor='rgba(244,67,54,0.1)'
), row=2, col=1)

fig.update_layout(
    title='<b>Rolling Variance Over Time (30-day window)</b>'
          '<br><sup>Spikes = high-volatility periods; post-2019 collapse reflects lower revenue, not stability</sup>',
    width=1100, height=600, hovermode='x unified'
)
fig.update_yaxes(title_text='Variance', row=1, col=1)
fig.update_yaxes(title_text='Variance', row=2, col=1)
fig.update_xaxes(title_text='Date', row=2, col=1)
fig.show()


**Descriptive:** Revenue variance grew from ~3–5T in 2012 to a peak of ~20T around mid-2018. A sharp structural break occurs post-2019 where variance collapses and stays below 5T. COGS mirrors this pattern almost exactly.

**Diagnostic:** High variance in 2015–2018 corresponds to the highest revenue period — peak revenue creates more room for daily swings. The post-2019 variance collapse is not operational improvement; it is a consequence of suppressed demand with less room to swing.

**Predictive:** The 2023–2024 test period will continue the low-variance regime. Lower variance means the model has a better chance at low MAE — but any surprise demand recovery could trigger spikes the model won't anticipate.

**Prescriptive:** Do not interpret low post-2019 variance as maturity. Before attempting revenue recovery, invest in dynamic inventory systems — the 2018 peak variance of 20T suggests the business was operationally overwhelmed during its previous growth phase.


### Chart 12 — Coefficient of Variation `[D → Px]`

In [ ]:
sales_sorted['rev_mean']  = sales_sorted['Revenue'].rolling(WINDOW).mean()
sales_sorted['cogs_mean'] = sales_sorted['COGS'].rolling(WINDOW).mean()
sales_sorted['rev_std']   = sales_sorted['Revenue'].rolling(WINDOW).std()
sales_sorted['cogs_std']  = sales_sorted['COGS'].rolling(WINDOW).std()
sales_sorted['rev_cv']    = sales_sorted['rev_std']  / sales_sorted['rev_mean']
sales_sorted['cogs_cv']   = sales_sorted['cogs_std'] / sales_sorted['cogs_mean']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sales_sorted['Date'], y=sales_sorted['rev_cv'],
    mode='lines', name='Revenue CV', line=dict(color='#2196F3', width=1.5)
))
fig.add_trace(go.Scatter(
    x=sales_sorted['Date'], y=sales_sorted['cogs_cv'],
    mode='lines', name='COGS CV', line=dict(color='#F44336', width=1.5)
))
fig.add_hline(y=1.0, line_dash='dash', line_color='grey',
              annotation_text='CV = 1 (std = mean)', opacity=0.6)
fig.update_layout(
    title='<b>Coefficient of Variation — Revenue vs COGS (30-day window)</b>'
          '<br><sup>CV = std/mean | scale-independent | Revenue and COGS track identically → COGS is demand-driven</sup>',
    xaxis_title='Date', yaxis_title='Coefficient of Variation',
    width=1100, height=450, hovermode='x unified',
    legend=dict(orientation='h', y=1.08)
)
fig.show()


**Descriptive:** Both Revenue CV and COGS CV oscillate persistently in the 0.2–0.7 range across 10 years, never approaching CV=1. The two lines are near-identical throughout, with only brief divergences around 2014–2015 and 2018.

**Diagnostic:** The near-perfect overlap confirms COGS is a fixed proportion of Revenue — costs scale identically with sales at the daily level. The business has never successfully decoupled its cost structure from demand volatility in 10 years.

**Predictive:** The 2023–2024 period will continue in the 0.3–0.6 CV range. Daily Revenue and COGS will deviate ~30–60% from their rolling average on any given day — an inherently wide prediction interval for any model.

**Prescriptive:** The stable Revenue/COGS ratio means **predicting COGS as a function of predicted Revenue** may outperform training a separate COGS model. Use `predicted_COGS = predicted_Revenue × mean(COGS/Revenue)` as a strong baseline. Additionally, negotiate fixed-price supplier contracts for top Streetwear products to decouple COGS volatility from revenue swings.
